# Run Confluence on an HPC

# Requirements
* GitHub and HPC account
* singularity or apptainer installed on your HPC
* Basic python environment

### BEFORE NOTEBOOK
1. Git fork all of the repos/modules you want to run (all branches, uncheck main only) 

### OVERALL TASKS IN THIS NOTEBOOK
1. Git clone all of the repos you want to run
2. Prep an empty_mnt directory to store confluence run (requires gdown package in environment)
3. Run image prep function to create the images from GitHub and your cloned modules
4. Create SLURM submission scripts for each module
5. Run the Confluence Driver Script Generator section of this notebook on your HPC to create a SLURM submission script that runs each of the modules one by one (the one click run)

## Functions (IGNORE)

In [ ]:
import os
import re
import shutil
import subprocess as sp
from pathlib import Path
import json
import glob
import numpy as np
import pandas as pd
import tarfile

# ============================================================
# Helpers
# ============================================================


def _validate_dir(dir: str | Path) -> Path:
    """Coerce str -> Path and validate type. Use at every entry point."""
    if isinstance(dir, str):
        dir = Path(dir)
    elif not isinstance(dir, Path):
        raise TypeError(
            f"Argument must be a Path or str. Got {type(dir)}."
        )
    return dir

# ============================================================
# Repo cloning
# ============================================================

def clone_repos(
    github_name: str,
    repo_dir: str | Path,
    repo_names: list,
    name_map: dict,
    branch: str | dict = "main",
):
    """Clone repositories with specified branch.
 
    Parameters
    ----------
    github_name : str
        GitHub username or organization name
    repo_dir : str or Path
        Directory to clone repos into
    repo_names : list
        List of repository names to clone
    name_map : dict
        Shorthand-name -> actual-repo-name mapping
    branch : str or dict, optional
        Branch name. Can be a string (same for all) or dict (per-repo).
    """
    repo_dir = _validate_dir(repo_dir)
    repo_dir.mkdir(parents=True, exist_ok=True)
    for name in repo_names:
        path = repo_dir / name
        repo_name = name_map.get(name, name)
        branch_val = branch.get(name, "main") if isinstance(branch, dict) else branch
        
        # NEW: handle 'org:branch' syntax for forks
        if ':' in branch_val:
            custom_org, actual_branch = branch_val.split(':', 1)
            url = f"https://github.com/{custom_org}/{repo_name}.git"
            branch_name = actual_branch
        else:
            url = f"https://github.com/{github_name}/{repo_name}.git"
            branch_name = branch_val
        
        # rest of function unchanged
        if path.exists():
            print(f"[Remove] Deleting existing {name} to overwrite...")
            shutil.rmtree(path)
        print(f"[Clone] Cloning {name} from branch {branch_name}...")
        sp.run(["git", "clone", "--branch", branch_name, url, name], cwd=repo_dir)


# ============================================================
# Singularity definition file generation
# Auto-discovery: walks the repo and overrides ALL local script files
# into the SIF, so edits to cloned repos flow into the next build
# without needing a docker push.
# ============================================================
def _create_lakeflow_defs(mod_dir: Path, tag: str) -> list[Path]:
    """Lakeflow has two Dockerfiles (input + deploy) → two SIFs."""
    sub_images = [
        ("lakeflow_input", "Dockerfile_input"),
        ("lakeflow_deploy", "Dockerfile_deploy"),
    ]
    created = []
    for sub_name, dockerfile_name in sub_images:
        dockerfile_path = mod_dir / dockerfile_name
        if not dockerfile_path.exists():
            print(f"lakeflow: {dockerfile_name} not found, skipping {sub_name}")
            continue

        # Same auto-discovery / entrypoint parsing as main function
        content = dockerfile_path.read_text()
        entrypoint = None
        for line in content.split("\n"):
            line = line.strip()
            if line.startswith("ENTRYPOINT") and "[" in line:
                matches = re.findall(r'"([^"]*)"', line)
                if matches:
                    entrypoint = " ".join(matches)

        override_extensions = {".py", ".sh", ".bash", ".r", ".R",
                               ".pl", ".json", ".yaml", ".yml", ".cfg", ".toml", ".ini"}
        skip_dirs = {".git", "__pycache__", ".github", ".eggs",
                     "env", "venv", ".mypy_cache", ".pytest_cache"}
        skip_files = {"Dockerfile", "Dockerfile_input", "Dockerfile_deploy",
                      "Singularity.def", "requirements.txt",
                      ".gitignore", ".dockerignore", "setup.py", "setup.cfg",
                      "pyproject.toml", "LICENSE", "README.md"}

        override_files = []
        for root, dirs, files in os.walk(mod_dir):
            dirs[:] = [d for d in dirs if d not in skip_dirs and not d.startswith(".")]
            for fname in files:
                if fname in skip_files:
                    continue
                ext = Path(fname).suffix.lower()
                if ext in override_extensions:
                    full_path = Path(root) / fname
                    rel_path = full_path.relative_to(mod_dir)
                    override_files.append(str(rel_path))

        def_content = f"""Bootstrap: docker
From: ghcr.io/swot-confluence/{sub_name}:{tag}

%files
"""
        for rel_path in sorted(override_files):
            src_full = mod_dir / rel_path
            if src_full.is_dir():
                def_content += f"    {rel_path}/. /app/{rel_path}\n"
            else:
                def_content += f"    {rel_path} /app/{rel_path}\n"

        def_content += "\n%post\n"
        def_content += f'    echo "=== {sub_name}: {len(override_files)} local files overridden ==="\n'

        if entrypoint:
            def_content += f'\n%runscript\n    exec {entrypoint} "$@"\n'

        def_path = mod_dir / f"Singularity_{sub_name}.def"
        def_path.write_text(def_content)
        print(f"{sub_name}: Singularity.def created ({len(override_files)} local files)")
        created.append(def_path)

    return created

def create_singularity_def(mod: str, repo_dir: str | Path, tag: str = "latest") -> Path | None:
    """Generate Singularity.def from Dockerfile, copying ALL local script files
    into /app/ to override whatever the base GHCR image had.

    This is the key advantage over Dockerfile-COPY parsing: any new file
    added to the cloned repo is picked up automatically on the next build.

    Also reinstalls the module's requirements.txt inside the SIF if present,
    which catches cases (notably MOI) where the base GHCR image is missing
    Python packages the module needs at runtime.
    """
    repo_dir = _validate_dir(repo_dir)
    mod_dir = repo_dir / mod
    
    IMAGE_NAME_MAP = {
        'hivdi': 'h2ivdi',
        # Add more as you discover them
    }
    image_name = IMAGE_NAME_MAP.get(mod, mod)

    # Special handling: lakeflow has two Dockerfiles → two SIFs
    if mod == "lakeflow":
        return _create_lakeflow_defs(mod_dir, tag)

    dockerfile_path = mod_dir / "Dockerfile"
    def_path = mod_dir / "Singularity.def"

    if not dockerfile_path.exists():
        print(f"{mod}: Dockerfile not found, skipping")
        return None

    # Parse Dockerfile for ENTRYPOINT
    content = dockerfile_path.read_text()
    entrypoint = None
    for line in content.split("\n"):
        line = line.strip()
        if line.startswith("ENTRYPOINT") and "[" in line:
            matches = re.findall(r'"([^"]*)"', line)
            if matches:
                entrypoint = " ".join(matches)

    # Discover ALL local script files to override
    override_extensions = {
        ".py", ".sh", ".bash", ".r", ".R",
        ".pl", ".json", ".yaml", ".yml", ".cfg", ".toml", ".ini",
    }
    skip_dirs = {
        ".git", "__pycache__", ".github", ".eggs",
        "env", "venv", ".mypy_cache", ".pytest_cache",
    }
    skip_files = {
        "Dockerfile", "Singularity.def", "requirements.txt",
        ".gitignore", ".dockerignore", "setup.py", "setup.cfg",
        "pyproject.toml", "LICENSE", "README.md",
    }

    override_files = []
    for root, dirs, files in os.walk(mod_dir):
        dirs[:] = [d for d in dirs if d not in skip_dirs and not d.startswith(".")]
        for fname in files:
            if fname in skip_files:
                continue
            ext = Path(fname).suffix.lower()
            if ext in override_extensions:
                full_path = Path(root) / fname
                rel_path = full_path.relative_to(mod_dir)
                override_files.append(str(rel_path))

    # Check for requirements.txt — used for both %files copy and %post pip install
    requirements_path = mod_dir / "requirements.txt"
    has_requirements = requirements_path.exists()

    # Build Singularity.def
    def_content = f"""Bootstrap: docker
From: ghcr.io/swot-confluence/{image_name}:{tag}

%files
"""

    # Explicit requirements.txt copy if present (for the %post pip install)
    # Note: requirements.txt is in skip_files for the auto-discovery loop above,
    # so we add it explicitly here only when we plan to reinstall it.
    if has_requirements:
        def_content += "    requirements.txt /app/requirements.txt\n"

    for rel_path in sorted(override_files):
        src_full = mod_dir / rel_path
        if src_full.is_dir():
            def_content += f"    {rel_path}/. /app/{rel_path}\n"
        else:
            def_content += f"    {rel_path} /app/{rel_path}\n"

    def_content += "\n%post\n"
    def_content += f'    echo "=== {mod}: {len(override_files)} local files overridden ==="\n'

    # Reinstall requirements.txt to catch any missing packages in the base image.
    # Critical for MOI which has had recurring missing-package issues. For modules
    # where the base image is already complete, pip detects packages are present
    # and the step is mostly a no-op.
    if has_requirements:
        def_content += f'    echo "=== {mod}: reinstalling requirements.txt ==="\n'
        def_content += "    if [ -f /app/requirements.txt ]; then\n"
        def_content += "        /app/env/bin/python3 -m pip install --no-cache-dir -r /app/requirements.txt 2>/dev/null || \\\n"
        def_content += "        python3 -m pip install --no-cache-dir -r /app/requirements.txt 2>/dev/null || \\\n"
        def_content += f'        echo "WARNING: pip install failed for {mod} — base image packages will be used"\n'
        def_content += "    fi\n"

    if mod == "output":
        def_content += """    # Fix nested output directory
    if [ -d /app/output/output ]; then
        cp -rf /app/output/output/* /app/output/
        rm -rf /app/output/output
    fi
"""

    if entrypoint:
        def_content += f"""
%runscript
    exec {entrypoint} "$@"
"""

    def_path.write_text(def_content)
    print(f"{mod}: Singularity.def created ({len(override_files)} local files)")
    return def_path

# ============================================================
# Per-module SLURM script generation
# ============================================================

def create_slurm_scripts(
    run_name: str,
    mnt_dir: str | Path,
    sif_dir: str | Path,
    sh_dir: str | Path,
    report_dir: str | Path,
    included_modules: list,
    sword_version: str = "17b",
    partition: str = "cpu-preempt",
    account: str = "pi_cjgleason_umass_edu",
    continue_downloads: bool = False,
):
    """Generate one .sh per module with SLURM headers and apptainer run command.
 
    Parameters
    ----------
    run_name : str
        Identifier for this run (used in job names and output paths)
    mnt_dir : Path
        Mount directory storing all confluence run data
    sif_dir : Path
        Directory containing built SIF files
    sh_dir : Path
        Directory to write .sh scripts
    report_dir : Path
        Directory for SLURM job logs
    included_modules : list
        Which modules to generate scripts for
    sword_version : str, optional
        SWORD version string passed to setfinder/combine_data (-s) and
        input/output (-v). Default '17b'. Common values: '16', '17', '17b'.
    partition : str, optional
        SLURM partition (default 'cpu-preempt')
    account : str, optional
        SLURM account (default 'pi_cjgleason_umass_edu')
    continue_downloads : bool, optional
        If True, pass -k flag to input module to skip already-downloaded files
    """
    
    mnt_dir = _validate_dir(mnt_dir)
    sif_dir = _validate_dir(sif_dir)
    sh_dir = _validate_dir(sh_dir)
    report_dir = _validate_dir(report_dir)
 
    submission_prefix = "#SBATCH"
    cpus_per_task = "1"
    skip_flag = " -k" if continue_downloads else ""
 
    job_details = {
        "partition": partition,
        "account": account,
        "nodes": "1",
        "cpus-per-task": cpus_per_task,
        "job-name": f"{run_name}_cfl",
    }
 
    # fmt: off

    command_dict = {
        "expanded_setfinder": f"apptainer run --bind {mnt_dir}/input:/data {sif_dir/'setfinder.sif'} -r reaches_of_interest.json -c continent.json -e -s {sword_version} -o /data -n /data -a MetroMan HiVDI SIC -i ${{SLURM_ARRAY_TASK_ID}}",
        "expanded_combine_data": f"apptainer run --bind {mnt_dir}/input:/data {sif_dir/'combine_data.sif'} -d /data -e -s {sword_version}",
        "input": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\nsingularity run --env SKIP_MOUNT_CHECK=true --bind {mnt_dir}/input:/mnt/data {sif_dir/'input.sif'} -v {sword_version} -r /mnt/data/expanded_reaches_of_interest.json -c SWOT_L2_HR_RiverSP_D -i ${{GLOBAL_INDEX}} -k -t '&start_time=2023-01-01T00:00:00Z&end_time=2027-12-31T23:59:59Z&'{skip_flag}",
        "non_expanded_setfinder": f"apptainer run --bind {mnt_dir}/input:/data {sif_dir/'setfinder.sif'} -c continent.json -s {sword_version} -o /data -n /data -a MetroMan HiVDI SIC -i ${{SLURM_ARRAY_TASK_ID}}",
        "non_expanded_combine_data": f"apptainer run --bind {mnt_dir}/input:/data {sif_dir/'combine_data.sif'} -d /data -s {sword_version}",
        "prediagnostics": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/diagnostics/prediagnostics:/mnt/data/output {sif_dir/'prediagnostics.sif'} -i ${{GLOBAL_INDEX}} -r reaches.json",
        "constrained_priors": f"apptainer run -c --writable-tmpfs --bind {mnt_dir}/input:/mnt/data {sif_dir/'priors.sif'} -i ${{SLURM_ARRAY_TASK_ID}} -r constrained -p usgs riggs -g -s local",
        "unconstrained_priors": f"apptainer run -c --writable-tmpfs --bind {mnt_dir}/input:/mnt/data {sif_dir/'priors.sif'} -i ${{SLURM_ARRAY_TASK_ID}} -r unconstrained -p usgs riggs -g -s local",
        "hivdi": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\nexport MPLBACKEND=agg\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/hivdi:/mnt/data/flpe/hivdi {sif_dir/'hivdi.sif'} /mnt/data/input/reaches.json --input-dir /mnt/data/input -i ${{SLURM_ARRAY_TASK_ID}}",
        "sic4dvar": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\nexport MPLBACKEND=agg\n\napptainer run --pwd /app --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/sic4dvar:/mnt/data/output,{mnt_dir}/logs:/mnt/data/logs {sif_dir/'sic4dvar.sif'} -r reaches.json --index ${{GLOBAL_INDEX}}",
        "busboi": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/busboi:/mnt/data/output {sif_dir/'busboi.sif'} -r reaches.json -i ${{GLOBAL_INDEX}}",
        "metroman": f'GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --env AWS_BATCH_JOB_ID="foo" --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/metroman:/mnt/data/output {sif_dir/"metroman.sif"} -i ${{GLOBAL_INDEX}} -r metrosets.json -s local -v',
        "metroman_consolidation": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/metroman:/mnt/data/flpe {sif_dir/'metroman_consolidation.sif'} -i ${{GLOBAL_INDEX}}",
        "unconstrained_momma": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/momma:/mnt/data/output {sif_dir/'momma.sif'} -r reaches.json -m 3 -i ${{GLOBAL_INDEX}}",
        "constrained_momma": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/momma:/mnt/data/output {sif_dir/'momma.sif'} -r reaches.json -m 3 -c -i ${{GLOBAL_INDEX}}",
        "sad": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/sad:/mnt/data/output {sif_dir/'sad.sif'} --reachfile reaches.json --index ${{GLOBAL_INDEX}}",
        "moi": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --env AWS_BATCH_JOB_ID='foo' --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe:/mnt/data/flpe,{mnt_dir}/moi:/mnt/data/output {sif_dir/'moi.sif'} -j basin.json -v -b unconstrained -i ${{GLOBAL_INDEX}}",
        "consensus": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe:/mnt/data/flpe {sif_dir/'consensus.sif'} --mntdir /mnt/data -r /mnt/data/input/reaches.json -i ${{GLOBAL_INDEX}}",
        "unconstrained_offline": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe:/mnt/data/flpe,{mnt_dir}/moi:/mnt/data/moi,{mnt_dir}/offline:/mnt/data/output {sif_dir/'offline.sif'} unconstrained timeseries integrator reaches.json ${{GLOBAL_INDEX}}",
        "lakeflow_input": f"apptainer run --pwd /app --bind {mnt_dir}/input:/mnt/data/input {sif_dir/'lakeflow_input.sif'} -c /mnt/data/input/reaches_of_interest.json -w 1 -i /mnt/data/input/lakeflow/ -s /mnt/data/input/sword/ -v {sword_version}",
        "lakeflow_deploy": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\napptainer run --pwd /app --env SLURM_ARRAY_TASK_ID=${{GLOBAL_INDEX}} --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe/lakeflow:/mnt/data/lakeflow_out {sif_dir/'lakeflow_deploy.sif'} -c /mnt/data/input/lakeflow/viable/viable_locations.csv -s /mnt/data/input/sos/ -w 6 -i /mnt/data/input/lakeflow/ -o /mnt/data/lakeflow_out/ -v {sword_version} --index=-512",        
        "validation": f"GLOBAL_INDEX=$(( ${{OFFSET:-0}} + SLURM_ARRAY_TASK_ID ))\n\nexport MPLBACKEND=agg\n\napptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe:/mnt/data/flpe,{mnt_dir}/moi:/mnt/data/moi,{mnt_dir}/offline:/mnt/data/offline,{mnt_dir}/validation:/mnt/data/output {sif_dir/'validation.sif'} -r reaches.json -t unconstrained -i ${{GLOBAL_INDEX}}",
        "output": f"apptainer run --bind {mnt_dir}/input:/mnt/data/input,{mnt_dir}/flpe:/mnt/data/flpe,{mnt_dir}/diagnostics:/mnt/data/diagnostics,{mnt_dir}/moi:/mnt/data/moi,{mnt_dir}/offline:/mnt/data/offline,{mnt_dir}/validation:/mnt/data/validation,{mnt_dir}/output:/mnt/data/output {sif_dir/'output.sif'} -s local -j /app/metadata/metadata.json -m input momma metroman sic4dvar hivdi busboi consensus validation lakeflow swot -v {sword_version} -i ${{SLURM_ARRAY_TASK_ID}}",
    }
    # fmt: on
 
    def _write_slurm_script(module_to_run: str, job_details: dict):
        submission_prefix = job_details["submission_prefix"]
        script_path = sh_dir / f"{module_to_run}.sh"
        with open(script_path, "w") as f:
            f.write("#!/bin/bash\n")
            f.write(f"{submission_prefix} -o {report_dir / f'{module_to_run}.%j_%a.out'}\n")
            for item in job_details:
                if item not in ("run_command", "module_name", "submission_prefix"):
                    f.write(f"{submission_prefix} --{item}={job_details[item]}\n")
            f.write(job_details["run_command"])
 
    for module_to_run, run_command in command_dict.items():
        if module_to_run == "moi":
            time_to_use = "00:30:00"
            mem_to_use = "2G"
        elif module_to_run == "lakeflow_input":
            time_to_use = "05:00:00"
            mem_to_use = "8G"
        elif module_to_run == "lakeflow_deploy":
            time_to_use = "12:00:00"
            mem_to_use = "60G"
            cpus_per_task= "6"
        elif module_to_run == "output":
            time_to_use = "05:00:00"
            mem_to_use = "4G"
        # lakeflow_input: time="05:00:00", mem="8G"  (downloading)
# lakeflow_deploy: time="12:00:00", mem="60G", cpus-per-task="6"  (Stan MCMC)
        else:
            time_to_use = "00:20:00"
            mem_to_use = "4G"
 
        if included_modules and module_to_run not in included_modules:
            continue
 
        print(f"DIRECTORY NAME: {run_name}\nMODULE: {module_to_run}")
 
        job_details.update({
            "run_command": run_command,
            "module_name": module_to_run,
            "mem": mem_to_use,
            "time": time_to_use,
            "cpus-per-task": cpus_per_task,
            "submission_prefix": submission_prefix,
            "job-name": f"{module_to_run}_{run_name}_cfl",
        })
 
        _write_slurm_script(module_to_run, job_details)



# ============================================================
# Driver script generation (with dry_run mode)
# ============================================================
                
def generate_slurm_driver(
    job_name: str,
    output_log_dir: str | Path,
    partition: str,
    time_limit: str,
    nodes: int,
    ntasks: int,
    cpus_per_task: int,
    mem: str,
    run: str,
    directory: str | Path,
    json_file: str | Path,
    expanded_json_file: str | Path,
    reach_json_file: str | Path,
    basin_json_file: str | Path,
    metroman_json_file: str | Path,
    batch_size: int,
    concurrent_jobs: int,
    script_jobs: dict,
    scripts: list,
    account: str = "pi_cjgleason_umass_edu",
    dry_run: bool = False,
) -> str:
    """Generate the driver script that sbatches each module in series.

    When dry_run=True, the generated script prints what it would submit
    without actually calling sbatch. Useful for validating the driver logic
    (job-count detection, batching, partition routing) before burning HPC time.

    To use dry_run: generate the script, then `bash slurm_driver.sh` it on
    the login node. It will use the same jq calls and same logic but only
    echo the sbatch commands.
    """
    slurm_header = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={output_log_dir}/{job_name}_%j_%a.out
#SBATCH --error={output_log_dir}/{job_name}_%j_%a.err
#SBATCH --partition={partition}
#SBATCH --account={account}
#SBATCH --time={time_limit}
#SBATCH --nodes={nodes}
#SBATCH --ntasks={ntasks}
#SBATCH --cpus-per-task={cpus_per_task}
#SBATCH --mem={mem}

run='{run}'
echo "Run: $run"
"""

    if dry_run:
        slurm_header += """
echo ""
echo "*** DRY RUN MODE — no jobs will be submitted ***"
echo ""
"""

    slurm_header += f"""
directory="{directory}"

# Parameters
json_file="{json_file}"
expanded_json_file="{expanded_json_file}"
reach_json_file="{reach_json_file}"
basin_json_file="{basin_json_file}"
metroman_json_file="{metroman_json_file}"
default_jobs=$(jq length "$json_file")

# Adjust to HPC requirements
batch_size={batch_size}
concurrent_jobs={concurrent_jobs}

# Map specific script names to their job counts
declare -A script_jobs=(
"""

    for script, jobs in script_jobs.items():
        slurm_header += f"    [{script}]={jobs}\n"
    slurm_header += ")\n\n"

    script_array = "    " + "\n    ".join(scripts)
    scripts_block = f"""scripts=(
{script_array}
)
"""

    # Submission block: branches on dry_run
    if dry_run:
        submission_block = r"""
        echo "[DRY RUN] Script:     $slurm_script"
        echo "[DRY RUN] Jobs:       $start to $end"
        echo "[DRY RUN] Array size: $((end - start + 1))"
        echo "[DRY RUN] Would run:  sbatch --export=ALL,OFFSET=${start} --array=0-$((end - start))%${concurrent_jobs} ${directory}/${slurm_script}"
        echo "---"
"""
    else:
        submission_block = r"""
        echo "Submitting jobs $start to $end from $slurm_script"
        job_id=$(sbatch --export=ALL,OFFSET=${start} --array=0-$((end - start))%${concurrent_jobs} "${directory}/${slurm_script}")
        job_id_number=$(echo $job_id | awk '{print $4}')

        echo "Waiting for job array $job_id_number to finish..."
        while squeue -j "$job_id_number" 2>/dev/null | grep -q "$job_id_number"; do
            job_info=$(squeue -j "${job_id_number}[]" --noheader -o "%i %T %R")
            held_tasks=$(echo "$job_info" | grep -i "requeued held" | awk '{print $1}')

            if [[ -n "$held_tasks" ]]; then
                echo "Detected held tasks in array $job_id_number:"
                echo "$held_tasks"
                for task in $held_tasks; do
                    echo "Cancelling task $task..."
                    scancel "$task"
                done
            fi

            sleep 10
        done

        echo "Batch $job_id_number has finished. Submitting next batch."
        date
"""

    finish_msg = (
        'echo "DRY RUN complete. No jobs were submitted."'
        if dry_run
        else 'echo "Run $run has finished successfully."'
    )

    body = rf"""{scripts_block}

for slurm_script in "${{scripts[@]}}"; do
    echo "Starting submission for: $slurm_script"
    date

    # Initialize num_jobs from script_jobs array FIRST
    num_jobs="${{script_jobs[$slurm_script]}}"

    # RETRY LOGIC: Dynamic job count for input.sh (handles any filesystem sync delay)
    if [[ "$slurm_script" == "input.sh" ]]; then
        max_retries=5
        retry_count=0
        expanded_jobs=""

        while [[ -z "$expanded_jobs" && $retry_count -lt $max_retries ]]; do
            if [[ -s "$expanded_json_file" ]]; then
                expanded_jobs=$(jq length "$expanded_json_file" 2>/dev/null)

                if [[ -n "$expanded_jobs" && "$expanded_jobs" -gt 0 ]]; then
                    echo "Found expanded_reaches_of_interest.json with $expanded_jobs reaches"
                    script_jobs["input.sh"]=$expanded_jobs
                    num_jobs=$expanded_jobs
                    break
                fi
            fi

            retry_count=$((retry_count + 1))
            if [[ $retry_count -lt $max_retries ]]; then
                echo "Waiting for expanded_reaches_of_interest.json... (attempt $retry_count/$max_retries)"
                sleep 5
            fi
        done

        if [[ -z "$expanded_jobs" || "$expanded_jobs" -eq 0 ]]; then
            echo "expanded_reaches_of_interest.json not found or empty, using reaches_of_interest.json"
            num_jobs=$(jq length "$json_file")
        fi
    fi

    # Dynamic job count for moi.sh
    if [[ -s "$basin_json_file" ]]; then
        basin_jobs=$(jq length "$basin_json_file")
        script_jobs["moi.sh"]=$basin_jobs
        if [[ "$slurm_script" == "moi.sh" ]]; then
            num_jobs=$basin_jobs
        fi
    fi

    # Dynamic job count for metroman.sh
    if [[ -s "$metroman_json_file" ]]; then
        metroman_jobs=$(jq length "$metroman_json_file")
        script_jobs["metroman.sh"]=$metroman_jobs
        if [[ "$slurm_script" == "metroman.sh" ]]; then
            num_jobs=$metroman_jobs
        fi
    fi

    # Dynamic job count for lakeflow_deploy.sh
    if [[ -s "${{mnt_dir}}/lakeflow/viable/lakeflow_lakes.json" ]]; then
        lakeflow_jobs=$(jq length "${{mnt_dir}}/lakeflow/viable/lakeflow_lakes.json")
        script_jobs["lakeflow_deploy.sh"]=$lakeflow_jobs
        if [[ "$slurm_script" == "lakeflow_deploy.sh" ]]; then
            num_jobs=$lakeflow_jobs
        fi
    fi

    # Fallback: use reaches.json if available, else reaches_of_interest.json
    if [[ -z "$num_jobs" || "$num_jobs" == "\$default_jobs" ]]; then
        if [[ -s "$reach_json_file" ]]; then
            num_jobs=$(jq length "$reach_json_file")
            echo "Using reach_json_file job count ($num_jobs) for $slurm_script"
        else
            num_jobs=$default_jobs
            echo "Using reaches_of_interest.json job count ($num_jobs) for $slurm_script"
        fi
    fi

    # Safety check
    if [[ -z "$num_jobs" ]]; then
        echo "Warning: No job count found for $slurm_script. Skipping."
        continue
    fi

    start=0
    while [ $start -lt $num_jobs ]; do
        end=$((start + batch_size - 1))
        if [ $end -ge $num_jobs ]; then
            end=$((num_jobs - 1))
        fi

        # Infinite-loop guard
        batch_count=$((end - start + 1))
        if [[ "$batch_count" -le 0 ]]; then
            echo "Error: Batch count is 0 (start=$start, end=$end). Terminating to prevent infinite loop."
            break
        fi
{submission_block}
        start=$((end + 1))
        sleep 5
    done

done

{finish_msg}
"""
    return slurm_header + body


## 1. Prepare Run

In [ ]:
BASE_DIR = Path('/nas/cee-water/cjgleason/ellie/SWOT/confluence/') #Path('/path/confluence/')        # directory storing confluence runs
REPO_DIR = BASE_DIR / 'modules/D' #BASE_DIR / 'modules'              # directory storing repos
RUN_NAME = 'svs17'                         # specific run name
SWORD_VERSION = '17b'                        # SWORD version: '16', '17', or '17b'

run_dir = BASE_DIR / f'confluence_{RUN_NAME}'
src_dir = BASE_DIR / 'confluence_empty'

partition = "cpu-preempt"
HPC_username = 'efriedmann_umass_edu' #'your_hpc_username'
github_name = 'efried130' #'your_github_name'
os.chdir(BASE_DIR)

print(run_dir)

In [ ]:
###############################
## INITIAL OR NEW MNT DOWNLOAD:
## RUN ONCE
###############################

## Install empty /mnt directory with input data and eventual output data
! pip install gdown
! gdown 15gmtZ_Gn27gaIDvmzeP_LVjAZekz_BSF

## Extract from tar.gz
tar_path = src_dir.with_suffix('.tar.gz')
with tarfile.open(tar_path, 'r:gz') as tar:
    tar.extractall(path=src_dir.parent)

In [ ]:
####################
## SUBSEQUENT RUNS:
###I.E. IF RUNNING ON NEW REACHES, RUN THIS, SKIP CELL ABOVE.
####################

src_dir = BASE_DIR / 'confluence_empty'  # initial unzipped gdown 
run_dir = BASE_DIR / f'confluence_{RUN_NAME}' # new directory for run
shutil.copytree(src_dir, run_dir) # copy the contents of empty to new (preserves initial data)

p = Path(f"{run_dir}/empty_mnt") # rename internal mnt to run name
p.rename(p.with_name(f"{RUN_NAME}_mnt"))



In [ ]:

# Point to necessary directories 
SIF_DIR = run_dir / 'sif' # Store built Docker images
sh_dir = run_dir / 'sh_scripts' # Store the sh scripts to run each module
report_dir = run_dir / 'report' # Job logs
mnt_dir = run_dir / f'{RUN_NAME}_mnt' #the mnt storing all confluence run data

os.environ['RUN_NAME'] = RUN_NAME
os.environ['BASE_DIR'] = str(BASE_DIR)
os.environ['REPO_DIR'] = str(REPO_DIR)
os.environ['SIF_DIR'] = str(SIF_DIR)
os.environ['APPTAINER_CACHEDIR'] = f'/work/{HPC_username}/.apptainer/cache' #add your hpc username here

# Fail safe for directory build
for d in [SIF_DIR, sh_dir, report_dir, REPO_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'RUN_NAME: {RUN_NAME}')
print(f'REPO_DIR: {REPO_DIR}')
print(f'SIF_DIR: {SIF_DIR}')

In [ ]:

#Name of confluence offline module
#expanded and non_expanded modules work from 'setfinder' and 'combine_data'
INCLUDED_MODULES = [
    'expanded_setfinder',
    'expanded_combine_data',
    'input',
    'non_expanded_setfinder',
    'non_expanded_combine_data',
    'prediagnostics',
    # # 'priors',
    'metroman',
    'metroman_consolidation',
    'unconstrained_momma',
    'busboi',
    'hivdi',
    # 'sad',
    'sic4dvar',
    'consensus',
    'moi',
    'unconstrained_offline',
    # 'lakeflow_input',
    # 'lakeflow_deploy',
    'validation',
    'output'
]

# Git modules to pull
TARGET_MODULES = [
    'setfinder',
    'combine_data',
    'input',
    'prediagnostics',
    # # 'priors',
    'metroman',
    'metroman_consolidation',
    'momma',
    "busboi",
    'hivdi',
    # 'sad',
    'sic4dvar',
    'consensus',
    'moi',
    # 'offline',
    # 'lakeflow',
    'validation',
    'output'
]

# Pull working branches for certain Git repos
# NOTE: can also clone another user's branch (i.e. 'output': 'someoneelse:my_feature_branch')
branch_map = {
    'setfinder': 'main',
    'combine_data': 'main',
    'input': 'main',
    'prediagnostics': 'main',
    # # 'priors': 'main',
    'metroman': 'main',
    'metroman_consolidation': 'main',
    'momma': 'main',
    'busboi': 'BUSBOI_main',
    'hivdi': 'main',
    # 'sad': 'main',
    'sic4dvar': 'main',
    'consensus': 'main',
    'moi': 'main',
    # 'offline': 'main',
    'lakeflow': 'efried130:hpc_lakeflow',
    'validation': 'main',
    'output': 'main'
}



## 2. Clone GitHub repositories

In [ ]:
name_map = {
        'offline': 'offline-discharge-data-product-creation',
        'moi': 'MOI',
        'validation': 'Validation',
        'hivdi': 'h2ivdi',
        'busboi': 'BUSBOI',
        'lakeflow': 'LakeFlow_Confluence',
    }
clone_repos(github_name=github_name, repo_dir=REPO_DIR, repo_names=TARGET_MODULES, name_map=name_map, branch=branch_map)

---
## 2b. Modify your reaches. Don't forget to change the reaches_of_interest.json file located in ./confluence_empty/empty_mnt/input/reaches_of_interest.json by adding your list of SWORD reaches (strings). Default: single reach for testing
---

## 3. Build SIF (apptainer)

In [ ]:
# Builds container and sif for all repos of interest

for mod in TARGET_MODULES:
    dockerfile = os.path.join(REPO_DIR, mod, 'Dockerfile')
    if os.path.exists(dockerfile):
        print(f"[{mod}]")
        with open(dockerfile) as f:
            lines = f.readlines()

            for line in lines:
                if line.strip().startswith(('COPY', 'ENTRYPOINT', 'FROM')):
                    print(line.strip())
        print()
    else:
        print(f'{mod} -> (Dockerfile x)')
        print()
        

for mod in TARGET_MODULES:
    create_singularity_def(mod, REPO_DIR, tag='latest')

# Build SIFs (special-case lakeflow's two sub-images)
for mod in TARGET_MODULES:
    if mod == "lakeflow":
        for sub_name in ("lakeflow_input", "lakeflow_deploy"):
            sif_path = os.path.join(SIF_DIR, f'{sub_name}.sif')
            def_file = f'Singularity_{sub_name}.def'
            print(f'{sub_name}: Building...')
            os.system(f'cd {REPO_DIR}/{mod} && apptainer build --force --ignore-fakeroot-command {sif_path} {def_file}')
    else:
        sif_path = os.path.join(SIF_DIR, f'{mod}.sif')
        print(f'{mod}: Building...')
        os.system(f'cd {REPO_DIR}/{mod} && apptainer build --force --ignore-fakeroot-command {sif_path} Singularity.def')

## 4. Create sh Scripts

In [ ]:
create_slurm_scripts(
    run_name=RUN_NAME,
    mnt_dir=mnt_dir,
    sif_dir=SIF_DIR,
    sh_dir=sh_dir,
    report_dir=report_dir,
    included_modules=INCLUDED_MODULES,
    sword_version=SWORD_VERSION,         
    partition=partition,              
    continue_downloads=False,             # set True to resume partial input downloads
)

## 5. Create Driver Script to run multiple modules
---
### Confluence Driver Script Generator (RUN ON HPC, NOT LOCALLY)
* Creates a batch submission script that will run all of your sif files in serial
* use sbatch to submit the entire run
* low resources and a long time should be used here, as all this job will do is launch your SLURM scripts you created for each module, it is basically a job manager

In [ ]:
# Create driver SLURM script for each run in run_list

# Define which modules have special (hardcoded) job counts
HARDCODED_JOBS = {
    "expanded_setfinder": "6",
    "expanded_combine_data": "1",
    "non_expanded_setfinder": "6",
    "non_expanded_combine_data": "1",
    "unconstrained_priors": "6",
    "constrained_priors": "6",
    "metroman_consolidation": "6",
    "lakeflow_input": "1",
    "output": "6",
}

# Define modules that need dynamic job counts (will use $default_jobs placeholder)
# These will be upgraded to specific JSON files during execution
DYNAMIC_MODULES = [
    "input",
    "prediagnostics",
    "metroman",
    "hivdi",
    "sic4dvar",
    "unconstrained_momma",
    "constrained_momma",
    "busboi",
    "sad",
    "moi",
    "consensus",
    "unconstrained_offline",
    "validation",
]



job_name = str(RUN_NAME)
output_log_dir = f"{run_dir}/log"
time_limit = "30:00:00"
nodes = 1
ntasks = 1
cpus_per_task = 1
mem = "10G"

directory = run_dir
sh_directory = f"{directory}/sh_scripts"
json_file = f"{directory}/{RUN_NAME}_mnt/input/reaches_of_interest.json"
expanded_json_file = f"{directory}/{RUN_NAME}_mnt/input/expanded_reaches_of_interest.json"
reach_json_file = f"{directory}/{RUN_NAME}_mnt/input/reaches.json"
basin_json_file = f"{directory}/{RUN_NAME}_mnt/input/basin.json"
metroman_json_file = f"{directory}/{RUN_NAME}_mnt/input/metrosets.json"


batch_size = 1000 # cluster specific
concurrent_jobs = 400 # cluster specific

# Make sure log dir exists
os.makedirs(output_log_dir, exist_ok=True)

# Dynamically build script_jobs based on INCLUDED_MODULES
script_jobs = {}
for module in INCLUDED_MODULES:
    script_name = f"{module}.sh"

    if module in HARDCODED_JOBS:
        # Use hardcoded job count
        script_jobs[script_name] = HARDCODED_JOBS[module]


# Dynamically build scripts list (same order as INCLUDED_MODULES)
scripts = [f"{module}.sh" for module in INCLUDED_MODULES]

# Common kwargs for both real and dry-run driver
driver_kwargs = dict(
    job_name=job_name,
    output_log_dir=output_log_dir,
    partition=partition,
    time_limit=time_limit,
    nodes=nodes,
    ntasks=ntasks,
    cpus_per_task=cpus_per_task,
    mem=mem,
    run=RUN_NAME,
    directory=sh_directory,
    json_file=json_file,
    expanded_json_file=expanded_json_file,
    reach_json_file=reach_json_file,
    basin_json_file=basin_json_file,
    metroman_json_file=metroman_json_file,
    batch_size=batch_size,
    concurrent_jobs=concurrent_jobs,
    script_jobs=script_jobs,
    scripts=scripts,
)

# Generate dry-run version for testing logic on login node
driver_dry = generate_slurm_driver(**driver_kwargs, dry_run=True)
with open(f"{sh_directory}/slurm_driver_dry.sh", "w") as f:
    f.write(driver_dry)
print(f"Wrote dry-run driver: {sh_directory}/slurm_driver_dry.sh")
print("  Test with: bash slurm_driver_dry.sh")

# Generate real driver
driver_real = generate_slurm_driver(**driver_kwargs, dry_run=False)
with open(f"{sh_directory}/slurm_driver.sh", "w") as f:
    f.write(driver_real)
print(f"Wrote real driver:    {sh_directory}/slurm_driver.sh")
print("  Submit with: sbatch slurm_driver.sh")



In [ ]:
## Optionally submit within notebook

## Run the dry-run driver directly on the login node (no SLURM submission)
## This will check job counts, JSON reach files are used correctly, script ordering, 
## path typos, batching math and save SLURM resources!

# result = sp.run(
#     ["bash", f"{sh_directory}/slurm_driver_dry.sh"],
#     capture_output=True,
#     text=True,
# )
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)
    
## Submit real run here
sp.run(["sbatch", f"{sh_directory}/slurm_driver.sh"], check=True)


---
# Reach or Module Changes

#### In order to run on different Type I reaches
* modify the file at /mnt/input/reaches_of_interest.json
* Create new directory (cont.json and prediag gets overwritten and changed each run!)

#### In order to change a module and test it:
### Overview
* change the module that you forked and cloned in HPC, re-build and test using this notebook, then push to GitHub. NOTE: IF YOU ARE A DEVELOPER, YOU MUST ALSO REBUILD THE CONTAINER ONCE CHANGES ARE PUSHED - SEE EXAMPLE 2
* Pay attention to and use tag names (custom_tag_name) for version control

### Details
* Use this notebook to generate everything in the HPC environment from cloning Git modules to running Confluence
* Docker images are built initially using GitHub container registry (ghcr.io/) and then overwritten with your HPC modules 
* This allows you to change module, re-build containers, and test as a module instantly
* Version control is handled by GitHub tag:
* Tag your local image
*      ! docker tag output:local ghcr.io/myGitAccount/moduleName:my-custom-tag
* Push to registry
*      docker push ghcr.io/myGitAccount/moduleName:my-custom-tag
* Modify tag name in function from 'latest' to 'my-custom-tag'

### Another Option
* Use a symlink to connect a previous run to a new directory (no need to build .sif files over and over)
* Run module of interest/script using the data in previous modules (only need to run the changed module!)
